In [1]:
# pip install yfinance pandas

from pathlib import Path
import pandas as pd
import yfinance as yf


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

START_DATE = "1990-01-01"
END_DATE = None          # Use None for all available data through the latest date
INTERVAL = "1d"

OUTPUT_DIR = Path("../Data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = OUTPUT_DIR / "german_stocks_daily_ohlcv.csv"


# ------------------------------------------------------------
# 2. Stock universe
# ------------------------------------------------------------

group_1 = {
    "Volkswagen_pref": "VOW3.DE",
    "Mercedes_Benz_Group": "MBG.DE",
    "Allianz": "ALV.DE",
    "Munich_Re": "MUV2.DE",
    "Bayer": "BAYN.DE",
    "BASF": "BAS.DE",
}

group_2 = {
    "Siemens": "SIE.DE",
    "EON": "EOAN.DE",
    "thyssenkrupp": "TKA.DE",
    "Lufthansa": "LHA.DE",
    "Merck_KGaA": "MRK.DE",
    "Henkel_pref": "HEN3.DE",
}

ticker_to_company = {
    **group_1,
    **group_2,
}

tickers = list(ticker_to_company.values())

ticker_to_company_name = {
    ticker: company
    for company, ticker in ticker_to_company.items()
}


# ------------------------------------------------------------
# 3. Download daily data from Yahoo Finance
# ------------------------------------------------------------

raw_data = yf.download(
    tickers=tickers,
    start=START_DATE,
    end=END_DATE,
    interval=INTERVAL,
    auto_adjust=False,
    actions=True,
    group_by="ticker",
    threads=True,
    progress=False,
)

if raw_data.empty:
    raise ValueError(
        "No data were downloaded. Check the ticker symbols, dates, "
        "internet connection, or Yahoo Finance availability."
    )


# ------------------------------------------------------------
# 4. Convert to a long CSV-friendly format
# ------------------------------------------------------------

data_frames = []

for ticker in tickers:

    if ticker not in raw_data.columns.get_level_values(0):
        print(f"Warning: no data returned for {ticker}.")
        continue

    ticker_data = raw_data[ticker].copy().dropna(how="all")

    if ticker_data.empty:
        print(f"Warning: {ticker} has no non-empty observations.")
        continue

    ticker_data.index.name = "date"
    ticker_data = ticker_data.reset_index()

    ticker_data.insert(1, "ticker", ticker)
    ticker_data.insert(
        2,
        "company",
        ticker_to_company_name[ticker]
    )

    data_frames.append(ticker_data)

if not data_frames:
    raise ValueError("No valid price series were returned.")


# ------------------------------------------------------------
# 5. Save one CSV file
# ------------------------------------------------------------

german_stocks = pd.concat(data_frames, ignore_index=True)

german_stocks.columns = [
    str(column).strip().lower().replace(" ", "_")
    for column in german_stocks.columns
]

german_stocks = german_stocks.sort_values(
    by=["ticker", "date"]
).reset_index(drop=True)

german_stocks.to_csv(
    OUTPUT_FILE,
    index=False
)

print(f"Download complete: {OUTPUT_FILE.resolve()}")
print(f"Rows saved: {len(german_stocks):,}")
print(f"Stocks included: {german_stocks['ticker'].nunique()}")

Download complete: /Users/stefangaman/Documents/ASE/Papers/MSR/Data/german_stocks_daily_ohlcv.csv
Rows saved: 88,284
Stocks included: 12
